# Variational Quantum Eigensolver (VQE) Básico

Vamos a implementar desde cero el corazón del NISQ: el bucle híbrido VQE. Utilizaremos el procesador cuántico de Qiskit para evaluar la energía (usando la primitiva `EstimatorV2`) y una CPU clásica con `SciPy` (`COBYLA`) para ajustar los parámetros.

El objetivo es encontrar la mínima energía de un sistema ficticio representado por un Hamiltoniano sencillo.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

## 1. Definimos el Hamiltoniano a minimizar
Usaremos un estado de interacción magnética Ising sencilla entre dos espines: $H = Z_0 \otimes Z_1 + X_0 + X_1$.

In [ ]:
hamiltonian = SparsePauliOp.from_list([("ZZ", 1.0), ("IX", 1.0), ("XI", 1.0)])
print("Hamiltoniano del sistema observador:\n", hamiltonian)

## 2. Preparar el Ansatz (Circuito Paramétrico)
Usaremos la plantilla heurística `TwoLocal` que alterna rotaciones individuales paramétricas (Ry) y enredos lineales continuos (CZ) para abarcar mucho mapa angulativo de Hilbert.

In [ ]:
ansatz = TwoLocal(num_qubits=2, rotation_blocks='ry', entanglement_blocks='cz', entanglement='linear', reps=2)
print("Ansatz Heurístico Moldeable:\n")
display(ansatz.decompose().draw('mpl'))

num_params = ansatz.num_parameters
print(f"El optimizador clásico tendrá que buscar los mejores {num_params} ángulos independientes para asfixiar a H.")

## 3. Construir la Función Costo Híbrida y Optimizar
Iniciamos el Estimator asíncrono para evaluar el autovalor.

In [ ]:
estimator = StatevectorEstimator()
history = []

def cost_function(params):
    # Encapsulamos la orden V2 (Pub Tuple)
    job = estimator.run([(ansatz, hamiltonian, [params])])
    res = job.result()
    
    energia = res[0].data.evs[0]
    history.append(energia)
    return energia

# Inicializamos basurilla azarosa angular clásica
initial_theta = np.random.rand(num_params) * 2 * np.pi

print("Desplegando loop Híbrido Cuántico Clásico...")
result_opt = minimize(cost_function, initial_theta, method='COBYLA', options={'maxiter': 200})

print("\n--- RESULTADO DE LA MAGIA VQE ---")
print(f"\nEnergía Final base aproximada alcanzada por QPU Estimaciones: {result_opt.fun:.4f}")

plt.plot(history, color='orange')
plt.title('Descenso Iterativo del Hamiltoniano Subyacente')
plt.xlabel('Pasos de la Iteración Clásica (Optimizador COBYLA)')
plt.ylabel('Valor Esperado Energía $E$')
plt.grid()
plt.show()